In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import joblib

Carga y preprocesado

In [2]:
BASE_FILE = 'dataset_imputed_with_festivo.csv'
df = pd.read_csv(BASE_FILE, parse_dates=['datetime'])
df = df.sort_values('datetime').set_index('datetime')

def winsorize(s, lower_pct=0.01, upper_pct=0.99):
    low, high = s.quantile([lower_pct, upper_pct])
    return s.clip(low, high)

df['precio_electricidad_MW'] = winsorize(df['precio_electricidad_MW'])
df['demanda_MW']            = winsorize(df['demanda_MW'])

In [3]:
df['hour']       = df.index.hour
df['sin_h']      = np.sin(2*np.pi*df['hour']/24)
df['cos_h']      = np.cos(2*np.pi*df['hour']/24)
df['dow']        = df.index.dayofweek
df['is_weekend'] = df['dow'].isin([5,6]).astype(int)
df['month']      = df.index.month
df['sin_m']      = np.sin(2*np.pi*(df['month']-1)/12)
df['cos_m']      = np.cos(2*np.pi*(df['month']-1)/12)
df['doy']        = df.index.dayofyear
df['sin_doy']    = np.sin(2*np.pi*(df['doy']-1)/365)
df['cos_doy']    = np.cos(2*np.pi*(df['doy']-1)/365)

# Rolling means (proxy de tendencia local)
df['price_roll3h']  = df['precio_electricidad_MW'].rolling(3).mean()
df['price_roll24h'] = df['precio_electricidad_MW'].rolling(24).mean()

# Lags de precio
df['price_lag1']    = df['precio_electricidad_MW'].shift(1)
df['price_lag24']   = df['precio_electricidad_MW'].shift(24)
df['price_lag168']  = df['precio_electricidad_MW'].shift(24*7)
df['price_lag8760'] = df['precio_electricidad_MW'].shift(24*365)

# Lags de demanda y generación
df['demand_lag168']     = df['demanda_MW'].shift(24*7)
df['gen_ren_lag168']    = df['generacion_renovable_MW'].shift(24*7)
df['gen_nonren_lag168'] = df['generacion_no_renovable_MW'].shift(24*7)

# Lag de temperatura
df['temp_lag168'] = df['temperatura_media'].shift(24*7)

# Quitar filas con NaNs para entrenar
df_model = df.dropna()

In [4]:
features = [
    'demanda_MW','generacion_renovable_MW','generacion_no_renovable_MW',
    'temperatura_media','festivo',
    'sin_h','cos_h','dow','is_weekend',
    'sin_m','cos_m','sin_doy','cos_doy',
    'price_roll3h','price_roll24h',
    'price_lag1','price_lag24','price_lag168','price_lag8760',
    'demand_lag168','gen_ren_lag168','gen_nonren_lag168',
    'temp_lag168'
]
X = df_model[features]
y = df_model['precio_electricidad_MW']

Split Train/Test (últimos 7 días de test)

In [5]:
split_date = df_model.index.max() - pd.Timedelta(days=7)
X_train = X.loc[:split_date]
y_train = y.loc[:split_date]
X_test  = X.loc[split_date + pd.Timedelta(hours=1):]
y_test  = y.loc[split_date + pd.Timedelta(hours=1):]

Entrenamiento

In [6]:
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {
    'n_estimators':  [200,500],
    'max_depth':     [6,10],
    'learning_rate': [0.05,0.1]
}
lgb = LGBMRegressor(random_state=42)
gscv = GridSearchCV(
    lgb, param_grid, cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1, verbose=1
)
gscv.fit(X_train, y_train)
model = gscv.best_estimator_
print("Mejores parámetros:", gscv.best_params_)

# Guardar modelo
joblib.dump(model, 'model_precio_lightgbm.pkl')

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004117 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4138
[LightGBM] [Info] Number of data points in the train set: 48720, number of used features: 23
[LightGBM] [Info] Start training from score 110.783589
Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 500}


['model_precio_lightgbm.pkl']

Evaluacion

In [7]:
y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = mean_absolute_percentage_error(y_test, y_pred)
def smape(a, f):
    return 100/len(a) * np.sum(2 * np.abs(f - a) / (np.abs(a) + np.abs(f)))
smape_val = smape(y_test.values, y_pred)

print(f"MAE test:   {mae:.3f}")
print(f"RMSE test:  {rmse:.3f}")
print(f"MAPE test:  {mape:.3%}")
print(f"SMAPE test: {smape_val:.3f}%")

MAE test:   1.467
RMSE test:  1.970
MAPE test:  1.027%
SMAPE test: 1.027%


Grafico últimos 7 días

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(y_test.index, y_test, label='Real (CSV)',    color='tab:orange')
plt.plot(y_test.index, y_pred, label='Predicción',    color='tab:blue')
plt.title('Predicción vs Real — últimos 7 días')
plt.ylabel('€/MWh'); plt.xlabel('Fecha')
plt.legend(); plt.grid(True)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %Hh'))
plt.xticks(rotation=45); plt.tight_layout()
plt.show()

Relacion demanda, Precio y Temperatura

In [ ]:
import matplotlib.pyplot as plt

# partimos de df completo con índice datetime
# ya tienes df (o df_model) que incluye las 3 columnas:
#   'demanda_MW', 'precio_electricidad_MW', 'temperatura_media'

fig, ax1 = plt.subplots(figsize=(14,4))

# a) Demanda (eje izquierdo)
ax1.fill_between(
    df.index, df['demanda_MW'],
    color='tab:orange', alpha=0.3, label='Demanda (MW)'
)
ax1.set_ylabel('Demanda (MW)', color='tab:orange')
ax1.tick_params(axis='y', labelcolor='tab:orange')

# b) Precio suavizado (mismo eje secundario Y a la derecha)
ax2 = ax1.twinx()
precio_smooth = df['precio_electricidad_MW'].rolling(24).mean()
ax2.plot(
    df.index, precio_smooth,
    color='tab:blue', label='Precio €/MWh (rolling 24h)'
)
ax2.set_ylabel('Precio €/MWh', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

# c) Temperatura (otro eje derecho, desplazado)
ax3 = ax1.twinx()
ax3.spines.right.set_position(("axes", 1.12))
ax3.plot(
    df.index, df['temperatura_media'],
    color='tab:green', alpha=0.6, label='Temperatura (°C)'
)
ax3.set_ylabel('Temp. (°C)', color='tab:green')
ax3.tick_params(axis='y', labelcolor='tab:green')

# leyenda combinada
lines, labels = [], []
for ax in (ax1, ax2, ax3):
    l, lab = ax.get_legend_handles_labels()
    lines += l; labels += lab
ax1.legend(lines, labels, loc='upper left', ncol=3)

# formato de fechas
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.title("Relación Demanda, Precio y Temperatura (serie completa)")
plt.tight_layout()
plt.show()
